# Feature Selection


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path("..").resolve()))

from src.config import (
    ARTIFACTS_DIR,
    ENGINEERED_DATA_FILE,
    FEATURE_COLUMNS_FILE,
    PROJECT_ROOT,
    TARGET_COLUMN,
)
from src.data.data_loader import load_engineered_data

NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / "10_feature_selection"
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "10_feature_selection"
SELECTED_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "creditcard_selected_features.csv"
SELECTED_FEATURES_FILE = ARTIFACTS_DIR / "selected_feature_names.json"

NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

print("Imports OK")
print(f"  Engineered data : {ENGINEERED_DATA_FILE}")
print(f"  Feature list    : {FEATURE_COLUMNS_FILE}")
print(f"  Tables dir      : {NOTEBOOK_TABLES_DIR}")
print(f"  Figures dir     : {NOTEBOOK_FIGURES_DIR}")
print(f"  Selected data   : {SELECTED_DATA_FILE}")
print(f"  Feature export  : {SELECTED_FEATURES_FILE}")


## 1. Objectives

- Measure how strongly each engineered feature relates to fraud.
- Detect redundant features that may duplicate the same signal.
- Add lightweight model-based evidence before deciding what moves into modeling.
- Build a final selected feature set with clear keep or drop decisions.
- Save tables and artifacts so the modeling notebook can start from a stable input set.


## Output Guide

This notebook writes its main outputs to:

- `reports/tables/10_feature_selection/`
- `reports/figures/10_feature_selection/`
- `data/processed/creditcard_selected_features.csv`
- `artifacts/selected_feature_names.json`

The final answer from this notebook is a feature-level decision table and a modeling-ready selected dataset.


## 2. Load Engineered Data


In [ ]:
df = load_engineered_data(ENGINEERED_DATA_FILE)
print(f"Engineered dataset shape: {df.shape}")
df.head()


## 3. Load Feature Metadata

Feature selection should stay connected to the feature-engineering outputs, so this section reloads the final feature list and the category labels created earlier.


In [ ]:
feature_columns = json.loads(FEATURE_COLUMNS_FILE.read_text(encoding="utf-8"))
feature_category_path = PROJECT_ROOT / "reports" / "tables" / "08_feature_engineering" / "final_feature_categories.json"

if feature_category_path.exists():
    feature_category_map = json.loads(feature_category_path.read_text(encoding="utf-8"))
else:
    print("WARNING: feature_category_map not found — all categories labeled UNKNOWN")
    feature_category_map = {feature: "UNKNOWN" for feature in feature_columns}

available_features = [feature for feature in feature_columns if feature in df.columns]
feature_metadata = pd.DataFrame({
    "feature": available_features,
    "feature_category": [feature_category_map.get(feature, "UNKNOWN") for feature in available_features],
})
feature_metadata.to_csv(NOTEBOOK_TABLES_DIR / "feature_metadata_overview.csv", index=False)
feature_metadata


## 4. Measure Feature Relevance

This section uses simple, interpretable signals to answer whether a feature looks useful before we think about redundancy:

- correlation with `Class`
- fraud vs non-fraud mean difference
- standardized separation between the two classes

These measures are not the final modeling decision, but they give us a strong first filter.


In [ ]:
relevance_rows = []
base_fraud_rate = df[TARGET_COLUMN].mean()

for feature in available_features:
    corr = df[feature].corr(df[TARGET_COLUMN])
    class_0 = df.loc[df[TARGET_COLUMN] == 0, feature]
    class_1 = df.loc[df[TARGET_COLUMN] == 1, feature]
    pooled_std = df[feature].std()
    standardized_mean_gap = abs(class_1.mean() - class_0.mean()) / pooled_std if pooled_std and not np.isnan(pooled_std) else 0.0
    relevance_rows.append({
        "feature": feature,
        "feature_category": feature_category_map.get(feature, "UNKNOWN"),
        "correlation_with_class": corr,
        "abs_correlation_with_class": abs(corr),
        "mean_class_0": class_0.mean(),
        "mean_class_1": class_1.mean(),
        "standardized_mean_gap": standardized_mean_gap,
        "base_fraud_rate": base_fraud_rate,
    })

feature_relevance_summary = pd.DataFrame(relevance_rows).sort_values(
    ["abs_correlation_with_class", "standardized_mean_gap"],
    ascending=False,
)
feature_relevance_summary.to_csv(NOTEBOOK_TABLES_DIR / "feature_relevance_summary.csv", index=False)
feature_relevance_summary.head(15)


## 5. Check Redundancy Among Features

Highly correlated engineered features can tell us the same story in slightly different ways. This section identifies feature pairs with strong overlap so we can avoid carrying duplicate signal into modeling.


In [ ]:
feature_corr_matrix = df[available_features].corr().abs()
upper_mask = np.triu(np.ones(feature_corr_matrix.shape), k=1).astype(bool)
upper_triangle = feature_corr_matrix.where(upper_mask)

redundancy_pairs = (
    upper_triangle.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_feature_correlation"})
    .sort_values("abs_feature_correlation", ascending=False)
)

REDUNDANCY_THRESHOLD = 0.85
flagged_redundancy_pairs = redundancy_pairs.loc[
    redundancy_pairs["abs_feature_correlation"] >= REDUNDANCY_THRESHOLD
].copy()
flagged_redundancy_pairs.to_csv(NOTEBOOK_TABLES_DIR / "feature_redundancy_pairs.csv", index=False)

redundancy_summary_rows = []
for feature in available_features:
    partner_series = upper_triangle[feature].dropna() if feature in upper_triangle.columns else pd.Series(dtype=float)
    reverse_partner_series = upper_triangle.loc[feature].dropna() if feature in upper_triangle.index else pd.Series(dtype=float)
    combined = pd.concat([partner_series, reverse_partner_series])
    if combined.empty:
        max_corr = 0.0
        strongest_partner = None
    else:
        strongest_partner = combined.idxmax()
        max_corr = combined.max()
    redundancy_summary_rows.append({
        "feature": feature,
        "strongest_partner": strongest_partner,
        "max_abs_feature_correlation": max_corr,
        "redundancy_flag": bool(max_corr >= REDUNDANCY_THRESHOLD),
    })

feature_redundancy_summary = pd.DataFrame(redundancy_summary_rows)
feature_redundancy_summary.to_csv(NOTEBOOK_TABLES_DIR / "feature_redundancy_summary.csv", index=False)
flagged_redundancy_pairs.head(15)


## 6. Add Model-Based Feature Signals

Statistical relevance is helpful, but feature selection should also ask whether simple models continue to find the same features important. This section adds two lightweight signals:

- Logistic Regression for linear separability after scaling
- Random Forest for nonlinear importance

The goal is not to finish modeling yet. It is to make the selection stage a little more model-aware.


In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    X = df[available_features]
    y = df[TARGET_COLUMN]

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    logistic_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]
    )
    logistic_pipeline.fit(X_train, y_train)
    logistic_abs_coef = np.abs(logistic_pipeline.named_steps["model"].coef_[0])

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    rf_model.fit(X_train, y_train)
    rf_importance = rf_model.feature_importances_

    model_feature_signal_summary = pd.DataFrame({
        "feature": available_features,
        "logistic_abs_coef": logistic_abs_coef,
        "random_forest_importance": rf_importance,
    })
except ModuleNotFoundError as exc:
    print(f"WARNING: model-based feature signals skipped because a dependency is missing: {exc}")
    model_feature_signal_summary = pd.DataFrame({
        "feature": available_features,
        "logistic_abs_coef": np.nan,
        "random_forest_importance": np.nan,
    })

model_feature_signal_summary = model_feature_signal_summary.sort_values(
    ["random_forest_importance", "logistic_abs_coef"],
    ascending=False,
)
model_feature_signal_summary.to_csv(NOTEBOOK_TABLES_DIR / "model_feature_signal_summary.csv", index=False)
model_feature_signal_summary.head(15)


## 7. Build Feature Selection Decision Table

The notebook now combines three angles:

- relevance to fraud
- redundancy with other features
- model-based usefulness

This creates an explicit keep or drop decision for each feature so the next notebook does not need to guess what belongs in the modeling dataset.


In [ ]:
selection_df = (
    feature_relevance_summary
    .merge(feature_redundancy_summary, on="feature", how="left")
    .merge(model_feature_signal_summary, on="feature", how="left")
)

score_columns = [
    "abs_correlation_with_class",
    "standardized_mean_gap",
    "logistic_abs_coef",
    "random_forest_importance",
]

for column in score_columns:
    rank_column = f"{column}_rank"
    selection_df[rank_column] = selection_df[column].rank(pct=True, ascending=True)

selection_df["combined_selection_score"] = selection_df[[f"{column}_rank" for column in score_columns]].mean(axis=1, skipna=True)
selection_df["selection_reason"] = "Retained as a candidate until final rule evaluation."
selection_df["selection_decision"] = "KEEP_MONITOR"

score_lookup = selection_df.set_index("feature")["combined_selection_score"].to_dict()
redundancy_drop_map = {}
for row in flagged_redundancy_pairs.itertuples(index=False):
    score_a = score_lookup.get(row.feature_a, 0.0)
    score_b = score_lookup.get(row.feature_b, 0.0)
    drop_feature = row.feature_a if score_a < score_b else row.feature_b
    keep_feature = row.feature_b if drop_feature == row.feature_a else row.feature_a
    existing = redundancy_drop_map.get(drop_feature)
    if existing is None or row.abs_feature_correlation > existing["abs_feature_correlation"]:
        redundancy_drop_map[drop_feature] = {
            "partner": keep_feature,
            "abs_feature_correlation": row.abs_feature_correlation,
        }

for idx, row in selection_df.iterrows():
    feature = row["feature"]
    score = row["combined_selection_score"]
    abs_corr = row["abs_correlation_with_class"]
    std_gap = row["standardized_mean_gap"]

    if feature in redundancy_drop_map:
        partner = redundancy_drop_map[feature]["partner"]
        selection_df.at[idx, "selection_decision"] = "DROP_REDUNDANCY"
        selection_df.at[idx, "selection_reason"] = (
            f"Highly correlated with {partner} and carries the weaker combined signal."
        )
    elif score >= 0.65 or abs_corr >= 0.08 or std_gap >= 0.5:
        selection_df.at[idx, "selection_decision"] = "KEEP"
        selection_df.at[idx, "selection_reason"] = "Shows strong fraud relevance and remains competitive after redundancy review."
    elif score >= 0.45 or abs_corr >= 0.03:
        selection_df.at[idx, "selection_decision"] = "KEEP_MONITOR"
        selection_df.at[idx, "selection_reason"] = "Useful enough to test in baseline models, but worth monitoring for stability."
    else:
        selection_df.at[idx, "selection_decision"] = "DROP_WEAK"
        selection_df.at[idx, "selection_reason"] = "Weak relative signal across statistical and model-based screens."

selection_df = selection_df.sort_values(
    ["selection_decision", "combined_selection_score", "abs_correlation_with_class"],
    ascending=[True, False, False],
)
selection_df.to_csv(NOTEBOOK_TABLES_DIR / "feature_selection_decisions.csv", index=False)
selection_df[[
    "feature",
    "feature_category",
    "abs_correlation_with_class",
    "standardized_mean_gap",
    "max_abs_feature_correlation",
    "logistic_abs_coef",
    "random_forest_importance",
    "combined_selection_score",
    "selection_decision",
    "selection_reason",
]].head(20)


## 8. Build Final Selected Dataset

The final dataset should include features that are either clearly retained or still strong enough to evaluate in baseline modeling. Dropped features remain documented in the decision table, but they do not move into the modeling-ready export.


In [ ]:
selected_feature_names = selection_df.loc[
    selection_df["selection_decision"].isin(["KEEP", "KEEP_MONITOR"]),
    "feature",
].tolist()

selected_feature_list = selection_df.loc[
    selection_df["selection_decision"].isin(["KEEP", "KEEP_MONITOR"]),
    ["feature", "feature_category", "selection_decision", "selection_reason"],
].reset_index(drop=True)

selected_dataset = df[selected_feature_names + [TARGET_COLUMN]].copy()

selected_feature_list.to_csv(NOTEBOOK_TABLES_DIR / "selected_feature_list.csv", index=False)
selected_dataset.to_csv(SELECTED_DATA_FILE, index=False)
SELECTED_FEATURES_FILE.write_text(json.dumps(selected_feature_names, indent=2), encoding="utf-8")

print(f"Selected feature count: {len(selected_feature_names)}")
print(f"Selected dataset shape: {selected_dataset.shape}")
selected_feature_list


## 9. Connection to Modeling and Decision System

Feature selection is the bridge between exploratory analysis and the first real modeling notebook. The decision table created here makes the next stage simpler:

- strong retained features can support the fraud probability score directly
- monitored features can be stress-tested in baseline models before they influence decision thresholds
- dropped features stay documented, which keeps the modeling pipeline explainable and reproducible

This also helps the future decision system because `BLOCK`, `REVIEW`, and `APPROVE` logic should rely on features that remain stable after redundancy checks and basic model validation.


## 10. Key Insights

- Feature selection should reward signal quality, not just feature quantity.
- Redundant engineered features can make a model harder to interpret without adding much predictive value.
- Model-aware checks help confirm whether a statistically interesting feature still matters once we approach the modeling stage.
- The final selected dataset should be treated as the default input for baseline model comparison.


## 11. Next Step

The next notebook should build baseline fraud-detection models on the selected feature set, compare threshold-sensitive metrics, and begin defining practical `BLOCK`, `REVIEW`, and `APPROVE` cutoffs.


In [ ]:
feature_selection_report = f"""# Feature Selection Report

## Key Findings

- This notebook combines univariate relevance, redundancy checks, and lightweight model-based signals to decide which engineered features move into modeling.
- Features are not selected by correlation alone; overlap between features is reviewed so the final set stays informative without unnecessary duplication.
- Logistic Regression and Random Forest are used as early model-aware checks to see whether the same features remain important once we move closer to modeling.
- The final output is a modeling-ready selected dataset plus an explicit keep or drop decision for every engineered feature.

## Feature Selection Logic

- Relevance is measured through correlation with `Class` and standardized fraud vs non-fraud separation.
- Redundancy is flagged through pairwise absolute feature correlation so duplicate signals can be pruned.
- Model-based evidence is added using Logistic Regression coefficients and Random Forest importances.
- Features are assigned to `KEEP`, `KEEP_MONITOR`, `DROP_REDUNDANCY`, or `DROP_WEAK` so the next notebook has a clear starting point.

## Connection to Modeling and Decision System

- Retained features form the default input space for baseline fraud models.
- Monitored features can still be tested in sensitivity analysis before they influence decision thresholds.
- Removing redundant or weak features makes downstream model behavior easier to explain and calibrate.
- A cleaner feature set supports more stable `BLOCK`, `REVIEW`, and `APPROVE` rules in the later decision system.

## Saved Tables

- `reports/tables/10_feature_selection/feature_metadata_overview.csv`
- `reports/tables/10_feature_selection/feature_relevance_summary.csv`
- `reports/tables/10_feature_selection/feature_redundancy_pairs.csv`
- `reports/tables/10_feature_selection/feature_redundancy_summary.csv`
- `reports/tables/10_feature_selection/model_feature_signal_summary.csv`
- `reports/tables/10_feature_selection/feature_selection_decisions.csv`
- `reports/tables/10_feature_selection/selected_feature_list.csv`
- `reports/tables/10_feature_selection/feature_selection_report.md`

## Saved Artifacts

- `data/processed/creditcard_selected_features.csv`
- `artifacts/selected_feature_names.json`
"""

report_path = NOTEBOOK_TABLES_DIR / "feature_selection_report.md"
report_path.write_text(feature_selection_report, encoding="utf-8")
print(f"Feature selection report saved to: {report_path}")
